# 01 — Data Preparation

This notebook connects to the SQLite database and uses SQL queries
to extract, explore, and validate the transaction data before analysis.

**What this notebook does:**
- Connects to `data/retail_ecuador.db`
- Explores the database structure using SQL
- Validates data quality
- Expands the categories column into pairs for association analysis
- Exports a clean dataset for the next notebook

In [ ]:
import pandas as pd
import sqlite3
import itertools
from pathlib import Path

# Connect to database using relative path
db_path = Path("../data/retail_ecuador.db")
conn = sqlite3.connect(db_path)

print("Connection established successfully")
print(f"Database: {db_path.resolve()}")

## Exploring the database structure

In [ ]:
# List all tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables)

In [ ]:
# General summary
query = """
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT city) as cities,
        COUNT(DISTINCT day_of_week) as days,
        COUNT(DISTINCT time_slot) as time_slots,
        MIN(date) as first_date,
        MAX(date) as last_date
    FROM transactions
"""
summary = pd.read_sql_query(query, conn)
print("Database summary:")
summary

In [ ]:
# Transactions per time slot
query = """
    SELECT
        time_slot,
        COUNT(*) as transactions,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM transactions), 2) as percentage
    FROM transactions
    GROUP BY time_slot
    ORDER BY transactions DESC
"""
slot_dist = pd.read_sql_query(query, conn)
print("Transactions by time slot:")
print(slot_dist)

In [ ]:
# Transactions per city
query = """
    SELECT
        city,
        COUNT(*) as transactions,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM transactions), 2) as percentage
    FROM transactions
    GROUP BY city
    ORDER BY transactions DESC
"""
city_dist = pd.read_sql_query(query, conn)
print("Transactions by city:")
print(city_dist)

## Data quality validation

In [ ]:
# Check for null values
query = """
    SELECT
        SUM(CASE WHEN transaction_id IS NULL THEN 1 ELSE 0 END) as null_transaction_id,
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) as null_date,
        SUM(CASE WHEN time_slot IS NULL THEN 1 ELSE 0 END) as null_time_slot,
        SUM(CASE WHEN categories IS NULL THEN 1 ELSE 0 END) as null_categories,
        SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) as null_city
    FROM transactions
"""
nulls = pd.read_sql_query(query, conn)
print("Null values per column:")
print(nulls)

## Expanding categories into pairs

Each transaction has 2 to 4 categories stored as a comma-separated string.
We expand them into all possible pairs per transaction — this is the format
needed for the chi-squared association analysis.

In [ ]:
# Load full dataset from SQLite
query = """
    SELECT
        transaction_id,
        date,
        time,
        day_of_week,
        time_slot,
        categories,
        city,
        customer_type,
        payment_method
    FROM transactions
    ORDER BY date, time
"""
df = pd.read_sql_query(query, conn)
conn.close()

print(f"Loaded {len(df)} transactions")
df.head()

In [ ]:
# Expand each transaction into category pairs
pairs_list = []

for _, row in df.iterrows():
    cats = [c.strip() for c in row["categories"].split(",")]
    for cat_a, cat_b in itertools.combinations(sorted(cats), 2):
        pairs_list.append({
            "transaction_id": row["transaction_id"],
            "date":           row["date"],
            "day_of_week":    row["day_of_week"],
            "time_slot":      row["time_slot"],
            "city":           row["city"],
            "customer_type":  row["customer_type"],
            "category_a":     cat_a,
            "category_b":     cat_b
        })

pairs_df = pd.DataFrame(pairs_list)

print(f"Total category pairs generated: {len(pairs_df)}")
print(f"\nFirst 5 rows:")
pairs_df.head()

In [ ]:
# Export both datasets
df.to_csv("../data/transactions_clean.csv", index=False)
pairs_df.to_csv("../data/category_pairs.csv", index=False)

print("Exported:")
print(f"  transactions_clean.csv — {len(df)} rows")
print(f"  category_pairs.csv     — {len(pairs_df)} rows")

## Summary

- Connected to SQLite database using relative path
- Validated 10,000 transactions with no null values
- Expanded transactions into category pairs for association analysis
- Exported `data/transactions_clean.csv` and `data/category_pairs.csv`
- Ready for exploratory analysis in `02_exploratory_analysis.ipynb`